# 🛡️ Dynamic Prompts, Tool Permissions & Human-in-the-Loop (HITL)

### Overview & Architecture
In production environments, agents must adhere to strict **security policies** and **safety constraints**:
1. **Dynamic Tool Permissions (RBAC)**: An agent should not have access to sensitive tools (such as reading an inbox or sending emails) until the user is authenticated.
2. **Dynamic System Prompts**: Adapting instructions on-the-fly based on user session state.
3. **Human-in-the-Loop (HITL) Safeguards**: Pausing execution before high-stakes actions (`send_email`), allowing human supervisors to inspect, approve, edit, or reject the action.
4. **Groq Acceleration**: Running the agent on Groq's high-speed **`llama-3.3-70b-versatile`** model with `InMemorySaver` state checkpointing.

## 📐 System Architecture: Middleware Pipeline & HITL Interception

The diagram below illustrates the middleware interception stack and human verification loop:

<div align="center">
  <img src="images/04_dynamic_hitl_pipeline.png" alt="Dynamic Middleware Pipeline & Human-in-the-Loop Safety Stack" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    User([👤 User Input]) --> Pipeline[⚙️ Agent Middleware Pipeline]

    subgraph Middleware [Middleware Interception Stack]
        direction TB
        M1[1. Dynamic Prompt Middleware<br/>Unauthenticated vs Authenticated Instructions]
        M2[2. Dynamic Tool Permission Middleware<br/>Filters Tools based on AuthenticatedState]
        M3[3. Model Invocation<br/>ChatGroq: llama-3.3-70b-versatile]
        M4[4. HumanInTheLoopMiddleware<br/>interrupt_on: send_email=True]
        M1 --> M2 --> M3 --> M4
    end

    Pipeline --> Middleware
    M4 -->|Safe Tool: check_inbox| AutoExec[⚡ Execute Tool]
    M4 -->|Sensitive Tool: send_email| InterruptState[(⏸️ Graph Paused<br/>__interrupt__ Payload)]

    InterruptState --> HumanReview{👤 Human Supervisor Inspection}
    HumanReview -->|Approve: Command resume| ExecEmail[📧 Execute send_email]
    HumanReview -->|Reject: Command resume| Abort[❌ Abort Action]
    ExecEmail --> Complete([✅ Operation Finished])
```

</details>


## 1. Environment & Warning Suppression

We load environment variables, verify the `GROQ_API_KEY`, and filter optional Pydantic serialization notices.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    print("⚠️ Warning: GROQ_API_KEY not found in environment.")
else:
    print("✅ GROQ_API_KEY loaded successfully.")

## 2. Defining Context & State Schemas

- **`EmailContext`**: Read-only static context holding system configuration (valid credentials).
- **`AuthenticatedState`**: Mutable agent state tracking whether the user has passed authentication (`authenticated: bool`).

In [ ]:
from dataclasses import dataclass
from langchain.agents import AgentState

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

class AuthenticatedState(AgentState):
    authenticated: bool

## 3. Defining Tools: Authentication, Inbox Reading & Email Sending

- `authenticate`: Validates credentials against `EmailContext`. Returns a `Command(update=...)` that sets `authenticated: True` or `False` in state.
- `check_inbox`: Read-only inbox retrieval tool.
- `send_email`: Write-action tool that dispatches an email (target for HITL interruption).

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails."""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response email."""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password."""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

## 4. Implementing Dynamic Middleware: Tool RBAC & Adaptive Prompts

- **`dynamic_tool_call` (`@wrap_model_call`)**: Dynamically overrides the tools presented to the LLM. If unauthenticated, only `[authenticate]` is provided; if authenticated, `[check_inbox, send_email]` are unlocked.
- **`dynamic_prompt` (`@dynamic_prompt`)**: Injects different system prompts based on the `authenticated` flag in state.

In [ ]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow read inbox and send email tools only if user is authenticated."""
    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt dynamically based on authentication status."""
    authenticated = request.state.get("authenticated")
    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

## 5. Compiling the Groq Agent with `HumanInTheLoopMiddleware`

We assemble the agent using `ChatGroq(model="llama-3.3-70b-versatile")` and register `HumanInTheLoopMiddleware` configured with:
```python
interrupt_on={"authenticate": False, "check_inbox": False, "send_email": True}
```
Whenever the agent decides to invoke `send_email`, execution halts and awaits external human confirmation.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

groq_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0
)

agent = create_agent(
    model=groq_model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            }
        )
    ]
)
print("✅ Dynamic HITL Agent compiled successfully with Groq.")

## 6. Execution Turn 1: Authentication & Inbox Checking

The user supplies credentials. The agent calls `authenticate`, switches into the authenticated prompt/tools, reads the inbox, and proposes email drafts.

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print("Agent Output (Turn 1):")
print("-" * 50)
print(response['messages'][-1].content)

## 7. Execution Turn 2: Triggering the HITL Breakpoint

The user instructs the agent to send a reply. The model generates a tool call for `send_email`, which triggers an immediate interrupt.

In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print("Execution halted at breakpoint.")
print(f"Interrupt status detected: {'__interrupt__' in response}")

## 8. Inspecting the Paused Action Request

The human supervisor inspects the pending action payload directly from `__interrupt__` to verify the email draft before any message is sent.

In [ ]:
action_request = response['__interrupt__'][0].value['action_requests'][0]
print("Pending Action Details:")
print(f"- Tool: {action_request['name']}")
print(f"- To:   {action_request['args'].get('to')}")
print(f"- Subj: {action_request['args'].get('subject')}")
print(f"- Body:\n{action_request['args'].get('body')}")

## 9. Resuming Execution with Human Approval (`Command(resume=...)`)

The human supervisor approves the action. We pass `Command(resume={"decisions": [{"type": "approve"}]})` using the same `thread_id` to continue execution.

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # Can be 'approve' or 'reject'
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print("Final Post-Approval Agent Message:")
print("-" * 50)
print(response["messages"][-1].content)